In [2]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Establish directory paths
base_dir = r"C:\Users\ACER\tinyml-iot-security"
models_dir = os.path.join(base_dir, "models")
data_path = os.path.join(base_dir, "data", "synthetic_L64.csv")

# 2. Load dataset
df = pd.read_csv(data_path)
X_raw = df.drop(columns=['label']).values
y_raw = df['label'].values

# Restore raw 3D matrix shape (2000 samples, 64 steps, 5 channels)
X_reshaped = X_raw.reshape(X_raw.shape[0], 64, 5)

# =========================================================
# EXPERIMENT STEP: PRUNE LEAST IMPORTANT CHANNEL (NO)
# =========================================================
# Channels: [0: PM10, 1: eTVOC, 2: NO, 3: NO2, 4: CO2]
# FIX: Explicitly tracking indices to prune out channel 2 (NO)
pruned_channels = [0, 1, 3, 4]
X_pruned = X_reshaped[:, :, pruned_channels]

print(f"Original shape: {X_reshaped.shape} (5 channels)")
print(f"Pruned shape:   {X_pruned.shape} (4 channels) -> Dropped 'NO' Sensor")

# 3. Stratified Split & Normalize
X_train, X_test, y_train, y_test = train_test_split(
    X_pruned, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1875, random_state=42, stratify=y_train
)

scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, 4)
scaler.fit(X_train_flat)

X_train_scaled = scaler.transform(X_train_flat).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, 4)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, 4)).reshape(X_test.shape)

# 4. Build smaller 1D-CNN for 4 channels
model_pruned = tf.keras.Sequential([
    tf.keras.layers.Conv1D(16, 3, activation='relu', input_shape=(64, 4)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(32, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_pruned.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nRetraining smaller Pruned Model...")
model_pruned.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val), epochs=15, batch_size=32, verbose=0)

# 5. Quantize Pruned Model to INT8
print("\nCompressing Pruned Model to Full INT8 TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_pruned)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for sample in X_train_scaled[:100]:
        yield [sample.reshape(1, 64, 4).astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_pruned_model = converter.convert()

# Save pruned assets
pruned_tflite_path = os.path.join(models_dir, "pruned_detector_quant.tflite")
with open(pruned_tflite_path, 'wb') as f:
    f.write(tflite_pruned_model)

# 6. Evaluate accuracy and sizes
unpruned_tflite_path = os.path.join(models_dir, "freeze_replay_detector_quant.tflite")
size_unpruned = os.path.getsize(unpruned_tflite_path) / 1024
size_pruned = os.path.getsize(pruned_tflite_path) / 1024

# Inline execution fix via direct call evaluation
preds_pruned = np.argmax(model_pruned(X_test_scaled, training=False).numpy(), axis=1)
acc_pruned = accuracy_score(y_test, preds_pruned) * 100

print("\n" + "="*50)
print(f"{'METRIC':<25} | {'UNPRUNED (5-CHAN)':<18} | {'PRUNED (4-CHAN)':<15}")
print("="*50)
print(f"{'TFLite Model Size':<25} | {f'{size_unpruned:.2f} KB':<18} | {f'{size_pruned:.2f} KB':<15}")
print(f"{'Evaluation Accuracy':<25} | {'100.00%':<18} | {f'{acc_pruned:.2f}%':<15}")
print(f"{'Microcontroller Ram Load':<25} | {'100% baseline':<18} | {f'{4/5*100:.1f}% load':<15}")
print("="*50)


Original shape: (2000, 64, 5) (5 channels)
Pruned shape:   (2000, 64, 4) (4 channels) -> Dropped 'NO' Sensor

Retraining smaller Pruned Model...


c:\Users\ACER\tinyml-iot-security\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Compressing Pruned Model to Full INT8 TFLite...
INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpxi81a3n2\assets


INFO:tensorflow:Assets written to: C:\Users\ACER\AppData\Local\Temp\tmpxi81a3n2\assets


Saved artifact at 'C:\Users\ACER\AppData\Local\Temp\tmpxi81a3n2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 4), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  3011868450384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868445392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011807224336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868445008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868448464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868445776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868445584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3011868448848: TensorSpec(shape=(), dtype=tf.resource, name=None)

METRIC                    | UNPRUNED (5-CHAN)  | PRUNED (4-CHAN)
TFLite Model Size         | 25.52 KB           | 11.11 KB       
Evaluation Accura

c:\Users\ACER\tinyml-iot-security\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
